# GAN-based Image Denoiser (PyTorch)

This notebook implements a conditional GAN (Pix2Pix-style) to map noisy images to clean images for denoising. It uses the repository's Dataset folder and synthesizes noise on-the-fly for robust training. It logs metrics, saves checkpoints, and includes an inference helper.

In [ ]:
# 1) Import Libraries and Set Seed
import os
import math
import random
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.utils import make_grid
from PIL import Image
import matplotlib.pyplot as plt

try:
    from skimage.metrics import peak_signal_noise_ratio as sk_psnr, structural_similarity as sk_ssim
except Exception as e:
    sk_psnr = None
    sk_ssim = None
    print("skimage not available; PSNR/SSIM will be limited.")

# Optional: Albumentations for stronger augmentations
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    _HAS_A = True
except Exception:
    _HAS_A = False

# Device and reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = (
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('mps') if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else
    torch.device('cpu')
)
print(f"Using device: {device}")

# Performance toggles for high-memory GPUs (CUDA)
if device.type == 'cuda':
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    except Exception:
        pass
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

torch.backends.cudnn.benchmark = True  # set False for fully deterministic


: 

In [ ]:
# 2) Configure Hyperparameters and Paths
from dataclasses import dataclass

@dataclass
class Config:
    # Data
    image_size: int = 256
    in_channels: int = 3
    base_channels: int = 64
    val_split: float = 0.1
    num_workers: int = 4

    # Training
    batch_size: int = 8
    epochs: int = 200
    lr: float = 2e-4
    betas: Tuple[float, float] = (0.5, 0.999)
    lambda_l1: float = 100.0
    mixed_precision: bool = True
    grad_clip_norm: float | None = 1.0
    accumulate_steps: int = 1  # set >1 to simulate larger batch size

    # Performance/DataLoader
    persistent_workers: bool = True  # requires num_workers > 0
    prefetch_factor: int = 2        # requires num_workers > 0
    drop_last: bool = True
    channels_last: bool = True

    # Noise synthesis
    noise_types: Tuple[str, ...] = ("gaussian", "poisson", "speckle", "jpeg")
    gaussian_sigma: Tuple[float, float] = (2.0, 25.0)  # pixel range [0,255]
    speckle_var: Tuple[float, float] = (0.001, 0.01)
    jpeg_quality: Tuple[int, int] = (10, 40)

    # Paths (relative to repo root)
    data_clean_dir: str = "Dataset/clean"
    output_dir: str = "Notebooks/checkpoints/gan_denoising"

cfg = Config()

# Resolve paths
repo_root = Path.cwd()
clean_dir = (repo_root / cfg.data_clean_dir).resolve()
out_dir = (repo_root / cfg.output_dir).resolve()
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Clean images dir: {clean_dir}")
print(f"Output dir:       {out_dir}")


In [ ]:
# 3) Prepare Dataset and Dataloaders (clean + synthetic noise)

def _to_uint8(img: np.ndarray) -> np.ndarray:
    img = np.clip(img, 0, 1)
    return (img * 255.0 + 0.5).astype(np.uint8)

class CleanWithSyntheticNoiseDataset(Dataset):
    def __init__(self, clean_root: Path, image_size: int = 256, use_albu: bool = False):
        self.clean_paths = []
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"):
            self.clean_paths.extend(sorted(clean_root.rglob(ext)))
        if not self.clean_paths:
            raise FileNotFoundError(f"No images found in {clean_root}")
        self.image_size = image_size
        self.use_albu = use_albu and _HAS_A
        if self.use_albu:
            self.albu_tf = A.Compose([
                A.LongestMaxSize(max_size=image_size),
                A.PadIfNeeded(image_size, image_size, border_mode=0, value=(0,0,0)),
                A.RandomCrop(image_size, image_size),
                A.HorizontalFlip(p=0.5),
            ])
        else:
            self.pil_tf = transforms.Compose([
                transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BILINEAR),
            ])

    def __len__(self):
        return len(self.clean_paths)

    def _read_rgb(self, path: Path) -> np.ndarray:
        with Image.open(path) as im:
            im = im.convert('RGB')
            return np.array(im).astype(np.float32) / 255.0

    def _synthesize_noise(self, clean_img: np.ndarray) -> np.ndarray:
        h, w, c = clean_img.shape
        noise_type = random.choice(cfg.noise_types)
        noisy = clean_img.copy()
        if noise_type == 'gaussian':
            sigma = random.uniform(*cfg.gaussian_sigma) / 255.0
            noisy = clean_img + np.random.normal(0.0, sigma, size=clean_img.shape).astype(np.float32)
        elif noise_type == 'poisson':
            vals = 2 ** np.ceil(np.log2(1.0 / 0.02))
            noisy = np.random.poisson(clean_img * vals).astype(np.float32) / vals
        elif noise_type == 'speckle':
            var = random.uniform(*cfg.speckle_var)
            noisy = clean_img + clean_img * np.random.normal(0.0, math.sqrt(var), size=clean_img.shape).astype(np.float32)
        elif noise_type == 'jpeg':
            # Re-compress with low quality
            pil = Image.fromarray(_to_uint8(clean_img))
            from io import BytesIO
            buf = BytesIO()
            quality = random.randint(*cfg.jpeg_quality)
            pil.save(buf, format='JPEG', quality=quality)
            buf.seek(0)
            noisy = np.array(Image.open(buf).convert('RGB')).astype(np.float32) / 255.0
        noisy = np.clip(noisy, 0.0, 1.0)
        return noisy

    def _to_tensor_norm(self, arr: np.ndarray) -> torch.Tensor:
        # Normalize to [-1,1]
        t = torch.from_numpy(arr.transpose(2,0,1)).float()
        t = t * 2.0 - 1.0
        return t

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        path = self.clean_paths[idx]
        clean = self._read_rgb(path)
        if self.use_albu:
            aug = self.albu_tf(image=clean)
            clean = aug['image']
        else:
            clean = np.array(self.pil_tf(Image.fromarray(_to_uint8(clean))))/255.0
        noisy = self._synthesize_noise(clean)
        return {
            'noisy': self._to_tensor_norm(noisy),
            'clean': self._to_tensor_norm(clean),
            'path': str(path)
        }

# Build datasets and loaders
full_ds = CleanWithSyntheticNoiseDataset(clean_dir, image_size=cfg.image_size, use_albu=True)
val_len = max(1, int(len(full_ds) * cfg.val_split))
train_len = len(full_ds) - val_len
train_ds, val_ds = random_split(full_ds, [train_len, val_len], generator=torch.Generator().manual_seed(seed))

def _make_loader(ds, bsz, shuffle):
    # persistent_workers and prefetch_factor require num_workers > 0
    pw = cfg.persistent_workers and (cfg.num_workers > 0)
    pf = cfg.prefetch_factor if cfg.num_workers > 0 else None
    return DataLoader(
        ds, batch_size=bsz, shuffle=shuffle, num_workers=cfg.num_workers, pin_memory=True,
        persistent_workers=pw, prefetch_factor=pf, drop_last=cfg.drop_last
    )

train_loader = _make_loader(train_ds, cfg.batch_size, True)
val_loader   = _make_loader(val_ds,   cfg.batch_size, False)

len(train_ds), len(val_ds)

In [ ]:
# 4) Preview Noisy vs Clean Samples

def denorm(t: torch.Tensor) -> torch.Tensor:
    # from [-1,1] to [0,1]
    return (t.clamp(-1,1) + 1.0) * 0.5

batch = next(iter(train_loader))
noisy_b = denorm(batch['noisy'])
clean_b = denorm(batch['clean'])
show_b = torch.cat([noisy_b[:4], clean_b[:4]], dim=0)
grid = make_grid(show_b, nrow=4)
plt.figure(figsize=(10,5))
plt.axis('off')
plt.title('Top: noisy | Bottom: clean')
plt.imshow(grid.permute(1,2,0).cpu().numpy())
plt.show()

In [ ]:
# 5) Define Generator (U-Net)
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, norm='in'):
        super().__init__()
        Norm = nn.InstanceNorm2d if norm=='in' else nn.BatchNorm2d
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            Norm(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            Norm(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch, norm='in'):
        super().__init__()
        self.pool = nn.AvgPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch, norm)
    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_ch, out_ch, bilinear=True, norm='in'):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_ch, out_ch, norm)
        else:
            self.up = nn.ConvTranspose2d(in_ch//2, in_ch//2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_ch, out_ch, norm)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Pad x1 to match x2 size
        diffY = x2.size(2) - x1.size(2)
        diffX = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diffX // 2, diffX - diffX//2, diffY // 2, diffY - diffY//2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1)
    def forward(self, x):
        return self.conv(x)

class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, base=64, bilinear=True, norm='in'):
        super().__init__()
        self.inc = DoubleConv(in_channels, base, norm)
        self.down1 = Down(base, base*2, norm)
        self.down2 = Down(base*2, base*4, norm)
        self.down3 = Down(base*4, base*8, norm)
        self.down4 = Down(base*8, base*8, norm)
        self.up1 = Up(base*16, base*4, bilinear, norm)
        self.up2 = Up(base*8, base*2, bilinear, norm)
        self.up3 = Up(base*4, base, bilinear, norm)
        self.up4 = Up(base*2, base, bilinear, norm)
        self.outc = OutConv(base, out_channels)
        self.tanh = nn.Tanh()
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        x = self.outc(x)
        return self.tanh(x)

gen = UNetGenerator(in_channels=cfg.in_channels, out_channels=cfg.in_channels, base=cfg.base_channels).to(device)
if cfg.channels_last and device.type == 'cuda':
    gen = gen.to(memory_format=torch.channels_last)
print(f"Generator params: {sum(p.numel() for p in gen.parameters())/1e6:.2f}M")

In [ ]:
# 6) Define PatchGAN Discriminator
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=2, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=stride, padding=1, bias=not norm)]
        if norm:
            layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=3, base=64):
        super().__init__()
        # Input is concatenation of noisy and target/generated: 2*in_channels
        self.model = nn.Sequential(
            # no norm in first block
            ConvBlock(in_channels*2, base, stride=2, norm=False),
            ConvBlock(base, base*2, stride=2, norm=True),
            ConvBlock(base*2, base*4, stride=2, norm=True),
            # stride=1 at the last conv before out
            ConvBlock(base*4, base*8, stride=1, norm=True),
            nn.Conv2d(base*8, 1, kernel_size=4, stride=1, padding=1)
        )
    def forward(self, x_noisy, x_target):
        x = torch.cat([x_noisy, x_target], dim=1)
        return self.model(x)

disc = PatchDiscriminator(in_channels=cfg.in_channels, base=cfg.base_channels).to(device)
if cfg.channels_last and device.type == 'cuda':
    disc = disc.to(memory_format=torch.channels_last)
print(f"Discriminator params: {sum(p.numel() for p in disc.parameters())/1e6:.2f}M")

In [ ]:
# 7) Define Losses (Adversarial + L1) and Optimizers
bce_logits = nn.BCEWithLogitsLoss().to(device)
l1_loss = nn.L1Loss().to(device)

opt_g = torch.optim.Adam(gen.parameters(), lr=cfg.lr, betas=cfg.betas)
opt_d = torch.optim.Adam(disc.parameters(), lr=cfg.lr, betas=cfg.betas)

scaler = torch.cuda.amp.GradScaler(enabled=(cfg.mixed_precision and device.type=='cuda'))

# Optional scheduler
scheduler_g = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=cfg.epochs)
scheduler_d = torch.optim.lr_scheduler.CosineAnnealingLR(opt_d, T_max=cfg.epochs)


In [ ]:
# 8) Training Step and Utility Functions (PSNR/SSIM)

def tensor_to_uint8(img_t: torch.Tensor) -> np.ndarray:
    # img_t in [-1,1]
    img = ((img_t.clamp(-1,1) + 1.0) * 0.5).detach().cpu().numpy()
    img = np.transpose(img, (0,2,3,1))  # N H W C
    img = (img * 255.0 + 0.5).astype(np.uint8)
    return img

@torch.no_grad()
def compute_psnr_ssim_batch(pred: torch.Tensor, target: torch.Tensor) -> Tuple[float, float]:
    if sk_psnr is None or sk_ssim is None:
        return float('nan'), float('nan')
    pred_np = tensor_to_uint8(pred)
    targ_np = tensor_to_uint8(target)
    psnrs, ssims = [], []
    for p, t in zip(pred_np, targ_np):
        psnrs.append(sk_psnr(t, p, data_range=255))
        try:
            ssims.append(sk_ssim(t, p, channel_axis=2, data_range=255))
        except TypeError:
            # Fallback for older skimage
            ssims.append(sk_ssim(t, p, multichannel=True, data_range=255))
    return float(np.mean(psnrs)), float(np.mean(ssims))


def train_step(batch):
    noisy = batch['noisy'].to(device, non_blocking=True)
    clean = batch['clean'].to(device, non_blocking=True)
    if cfg.channels_last and device.type == 'cuda':
        noisy = noisy.contiguous(memory_format=torch.channels_last)
        clean = clean.contiguous(memory_format=torch.channels_last)

    valid = torch.ones((noisy.size(0), 1, 30, 30), device=device)  # shape will be adjusted dynamically below
    fake = torch.zeros_like(valid)

    # Adjust valid/fake shape by running one disc forward on zeros to get map size
    with torch.no_grad():
        dshape = disc(noisy, clean).shape
    valid = torch.ones(dshape, device=device)
    fake = torch.zeros(dshape, device=device)

    # 1) Update Discriminator
    opt_d.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
        gen_clean = gen(noisy)
        d_real = disc(noisy, clean)
        d_fake = disc(noisy, gen_clean.detach())
        loss_d_real = bce_logits(d_real, valid)
        loss_d_fake = bce_logits(d_fake, fake)
        loss_d = (loss_d_real + loss_d_fake) * 0.5
    scaler.scale(loss_d).backward()
    if cfg.grad_clip_norm:
        scaler.unscale_(opt_d)
        torch.nn.utils.clip_grad_norm_(disc.parameters(), cfg.grad_clip_norm)
    scaler.step(opt_d)

    # 2) Update Generator
    opt_g.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
        gen_clean = gen(noisy)
        d_fake = disc(noisy, gen_clean)
        adv_loss = bce_logits(d_fake, valid)
        l1 = l1_loss(gen_clean, clean)
        loss_g = adv_loss + cfg.lambda_l1 * l1
    scaler.scale(loss_g).backward()
    if cfg.grad_clip_norm:
        scaler.unscale_(opt_g)
        torch.nn.utils.clip_grad_norm_(gen.parameters(), cfg.grad_clip_norm)
    scaler.step(opt_g)
    scaler.update()

    return {
        'loss_d': loss_d.item(),
        'loss_g': loss_g.item(),
        'adv': adv_loss.item(),
        'l1': l1.item(),
    }


In [ ]:
# 9) Train Loop with Checkpointing and Logging
import csv
from datetime import datetime

ckpt_dir = out_dir
(ckpt_dir / 'samples').mkdir(parents=True, exist_ok=True)
log_csv = ckpt_dir / 'epoch_results.csv'
log_txt = ckpt_dir / 'epoch_results.txt'

best_metric = -float('inf')

def save_checkpoint(path: Path, epoch: int):
    state = {
        'epoch': epoch,
        'gen': gen.state_dict(),
        'disc': disc.state_dict(),
        'opt_g': opt_g.state_dict(),
        'opt_d': opt_d.state_dict(),
        'scaler': scaler.state_dict() if scaler.is_enabled() else None,
        'cfg': vars(cfg),
    }
    torch.save(state, str(path))


def load_checkpoint(path: Path):
    state = torch.load(path, map_location=device)
    gen.load_state_dict(state['gen'])
    disc.load_state_dict(state['disc'])
    opt_g.load_state_dict(state['opt_g'])
    opt_d.load_state_dict(state['opt_d'])
    if state.get('scaler') and scaler.is_enabled():
        scaler.load_state_dict(state['scaler'])
    return state.get('epoch', 0)


def validate(epoch: int, max_batches: int = 10):
    gen.eval()
    losses_g, losses_d, psnrs, ssims = [], [], [], []
    with torch.no_grad():
        for bi, batch in enumerate(val_loader):
            if bi >= max_batches:
                break
            noisy = batch['noisy'].to(device)
            clean = batch['clean'].to(device)
            gen_clean = gen(noisy)
            d_fake = disc(noisy, gen_clean)
            d_real = disc(noisy, clean)
            loss_d = 0.5*(bce_logits(d_real, torch.ones_like(d_real)) + bce_logits(d_fake, torch.zeros_like(d_fake)))
            adv_loss = bce_logits(d_fake, torch.ones_like(d_fake))
            l1 = l1_loss(gen_clean, clean)
            loss_g = adv_loss + cfg.lambda_l1 * l1
            losses_g.append(loss_g.item())
            losses_d.append(loss_d.item())
            p, s = compute_psnr_ssim_batch(gen_clean, clean)
            if not np.isnan(p): psnrs.append(p)
            if not np.isnan(s): ssims.append(s)
    gen.train()
    return {
        'loss_g': float(np.mean(losses_g)) if losses_g else float('inf'),
        'loss_d': float(np.mean(losses_d)) if losses_d else float('inf'),
        'psnr': float(np.mean(psnrs)) if psnrs else float('nan'),
        'ssim': float(np.mean(ssims)) if ssims else float('nan'),
    }


def visualize_samples(epoch: int, batch, save: bool = True):
    gen.eval()
    with torch.no_grad():
        noisy = batch['noisy'].to(device)[:4]
        clean = batch['clean'].to(device)[:4]
        denoised = gen(noisy)
    show = torch.cat([denorm(noisy), denorm(denoised), denorm(clean)], dim=0)
    grid = make_grid(show, nrow=4)
    plt.figure(figsize=(10,8))
    plt.axis('off')
    plt.title(f'Epoch {epoch}: Row1 noisy | Row2 denoised | Row3 clean')
    plt.imshow(grid.permute(1,2,0).cpu().numpy())
    if save:
        img_path = ckpt_dir / 'samples' / f'epoch_{epoch:04d}.png'
        plt.savefig(img_path, bbox_inches='tight')
    plt.show()
    gen.train()

# Prepare CSV log
if not log_csv.exists():
    with open(log_csv, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','loss_g','loss_d','val_loss_g','val_loss_d','val_psnr','val_ssim','timestamp'])

start_epoch = 0
resume_path = ckpt_dir / 'last.pt'
if resume_path.exists():
    print(f"Resuming from: {resume_path}")
    start_epoch = load_checkpoint(resume_path) + 1

for epoch in range(start_epoch, cfg.epochs):
    gen.train(); disc.train()
    running = {'loss_g':0.0,'loss_d':0.0,'adv':0.0,'l1':0.0}
    for bi, batch in enumerate(train_loader):
        mets = train_step(batch)
        for k in running:
            running[k] += mets.get(k, 0.0)
        if (bi+1) % 100 == 0:
            msg = f"Epoch {epoch+1}/{cfg.epochs} Step {bi+1}: " + \
                  ", ".join([f"{k}={running[k]/(bi+1):.4f}" for k in running])
            print(msg)
    # Validation
    val_m = validate(epoch)

    # Logging
    with open(log_csv, 'a', newline='') as f:
        w = csv.writer(f)
        w.writerow([
            epoch,
            running['loss_g']/max(1,len(train_loader)),
            running['loss_d']/max(1,len(train_loader)),
            val_m['loss_g'], val_m['loss_d'], val_m['psnr'], val_m['ssim'],
            datetime.now().isoformat(timespec='seconds')
        ])
    with open(log_txt, 'a') as f:
        f.write(f"Epoch {epoch}: {val_m}\n")

    # Visualization (use a small val batch)
    try:
        batch_vis = next(iter(val_loader))
        visualize_samples(epoch, batch_vis, save=True)
    except StopIteration:
        pass

    # Save checkpoints
    save_checkpoint(ckpt_dir / 'last.pt', epoch)

    # Track best by SSIM, fallback to PSNR if NaN
    metric = val_m['ssim']
    if np.isnan(metric):
        metric = val_m['psnr'] if not np.isnan(val_m['psnr']) else -float('inf')
    if metric > best_metric:
        best_metric = metric
        save_checkpoint(ckpt_dir / 'best.pt', epoch)
        print(f"Saved new best checkpoint at epoch {epoch} with metric={best_metric:.4f}")

    # Step schedulers
    scheduler_g.step(); scheduler_d.step()

print("Training finished.")

In [ ]:
# 10) Validate and Visualize Results (on full val set)
@torch.no_grad()
def evaluate_full_val(save_images: bool = True):
    gen.eval()
    all_psnr, all_ssim = [], []
    for bi, batch in enumerate(val_loader):
        noisy = batch['noisy'].to(device)
        clean = batch['clean'].to(device)
        denoised = gen(noisy)
        p, s = compute_psnr_ssim_batch(denoised, clean)
        if not np.isnan(p): all_psnr.append(p)
        if not np.isnan(s): all_ssim.append(s)
        if save_images and bi < 3:
            show = torch.cat([denorm(noisy[:4]), denorm(denoised[:4]), denorm(clean[:4])], dim=0)
            grid = make_grid(show, nrow=4)
            plt.figure(figsize=(10,8))
            plt.axis('off')
            plt.title(f'Val batch {bi}: noisy | denoised | clean')
            plt.imshow(grid.permute(1,2,0).cpu().numpy())
            plt.show()
    mean_psnr = float(np.mean(all_psnr)) if all_psnr else float('nan')
    mean_ssim = float(np.mean(all_ssim)) if all_ssim else float('nan')
    print(f"Full Val -> PSNR: {mean_psnr:.3f}, SSIM: {mean_ssim:.4f}")
    return mean_psnr, mean_ssim

In [ ]:
# 11) Save, Load, and Export Model (TorchScript/ONNX)

def export_torchscript(ts_path: Path | str = None):
    ts_path = Path(ts_path) if ts_path else (ckpt_dir / 'generator_scripted.pt')
    gen.eval()
    dummy = torch.randn(1, cfg.in_channels, cfg.image_size, cfg.image_size, device=device)
    traced = torch.jit.trace(gen, dummy)
    traced.save(str(ts_path))
    print(f"Saved TorchScript to {ts_path}")


def export_onnx(onnx_path: Path | str = None):
    onnx_path = Path(onnx_path) if onnx_path else (ckpt_dir / 'generator.onnx')
    gen.eval()
    dummy = torch.randn(1, cfg.in_channels, cfg.image_size, cfg.image_size, device=device)
    torch.onnx.export(
        gen, dummy, str(onnx_path),
        input_names=['noisy'], output_names=['denoised'],
        dynamic_axes={'noisy': {0: 'batch'}, 'denoised': {0: 'batch'}},
        opset_version=13
    )
    print(f"Saved ONNX to {onnx_path}")

In [ ]:
# 12) Inference on Single Image from Disk
from torchvision.transforms.functional import to_tensor

@torch.no_grad()
def denoise_image(path_in: str | Path, path_out: str | Path = None):
    path_in = Path(path_in)
    if path_out is None:
        path_out = ckpt_dir / f"denoised_{path_in.stem}.png"
    else:
        path_out = Path(path_out)

    img = Image.open(path_in).convert('RGB')
    arr = np.array(img).astype(np.float32) / 255.0
    h, w, _ = arr.shape

    # Pad to multiple of 16 for U-Net
    def _pad_to_multiple(x, base=16):
        nh = (h + base - 1)//base*base
        nw = (w + base - 1)//base*base
        pad_h = nh - h
        pad_w = nw - w
        x = np.pad(x, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')
        return x, pad_h, pad_w

    arr_pad, ph, pw = _pad_to_multiple(arr, 16)
    t = torch.from_numpy(arr_pad.transpose(2,0,1)).float().unsqueeze(0)
    t = t*2-1
    t = t.to(device)

    gen.eval()
    out = gen(t)
    out = (out.clamp(-1,1)+1)/2.0
    out = out.squeeze(0).permute(1,2,0).detach().cpu().numpy()
    # Unpad
    out = out[:h, :w, :]
    out = (out*255.0+0.5).astype(np.uint8)
    Image.fromarray(out).save(path_out)
    print(f"Saved denoised image to {path_out}")

# Example usage (uncomment after training):
# denoise_image('Dataset/noisy/example.png')

In [ ]:
# 13) Lightweight Unit Tests for Shapes and Forward Pass
# These tests are safe to run before training to validate shapes.

def run_unit_tests():
    print("Running unit tests...")
    G = UNetGenerator(in_channels=cfg.in_channels, out_channels=cfg.in_channels, base=cfg.base_channels).to(device)
    D = PatchDiscriminator(in_channels=cfg.in_channels, base=cfg.base_channels).to(device)

    x = torch.randn(2, cfg.in_channels, cfg.image_size, cfg.image_size, device=device)
    y = torch.randn(2, cfg.in_channels, cfg.image_size, cfg.image_size, device=device)

    y_hat = G(x)
    assert y_hat.shape == x.shape, f"Generator output shape mismatch: {y_hat.shape} vs {x.shape}"

    d_out = D(x, y)
    assert d_out.ndim == 4 and d_out.shape[1] == 1, f"Discriminator shape invalid: {d_out.shape}"

    loss_fn = nn.BCEWithLogitsLoss()
    d_loss = loss_fn(d_out, torch.ones_like(d_out))
    assert torch.isfinite(d_loss), "Discriminator loss not finite"

    # One training step should update params
    optg = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.5,0.999))
    optd = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.5,0.999))

    # record a weight
    w_before = next(G.parameters()).detach().clone()

    # simple losses
    d_real = D(x, y)
    d_fake = D(x, G(x).detach())
    ld = 0.5*(loss_fn(d_real, torch.ones_like(d_real)) + loss_fn(d_fake, torch.zeros_like(d_fake)))
    optd.zero_grad(); ld.backward(); optd.step()

    d_fake = D(x, G(x))
    lg = loss_fn(d_fake, torch.ones_like(d_fake)) + 10.0*F.l1_loss(G(x), y)
    optg.zero_grad(); lg.backward(); optg.step()

    w_after = next(G.parameters()).detach().clone()
    assert not torch.allclose(w_before, w_after), "Weights did not update in unit test"
    print("All unit tests passed.")

# run_unit_tests()  # uncomment to run


## Resume training from checkpoint

- The notebook automatically resumes from `Notebooks/checkpoints/gan_denoising/last.pt` if it exists.
- To manually load `best.pt` before evaluating or inferring, run the following cell.


In [ ]:
# Load best checkpoint before validation/inference (optional)
# best_path = ckpt_dir / 'best.pt'
# if best_path.exists():
#     load_checkpoint(best_path)
#     print('Loaded best checkpoint.')
#     evaluate_full_val(save_images=True)


## Optional: Use existing paired noisy/clean dataset via `dataset_mapping.csv`
If you prefer training on real noisy/clean pairs instead of synthetic noise, you can use the mapping file in `Dataset/dataset_mapping.csv`. Uncomment and run the next cell to switch the dataloaders.

In [ ]:
# Switch to paired dataset from mapping (optional)
# import pandas as pd
#
# class PairedMappingDataset(Dataset):
#     def __init__(self, mapping_csv: Path, root: Path, image_size: int = 256):
#         self.df = pd.read_csv(mapping_csv)
#         self.root = root
#         self.image_size = image_size
#         self.tf = transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BILINEAR)
#     def __len__(self):
#         return len(self.df)
#     def _read_rgb(self, path: Path) -> np.ndarray:
#         with Image.open(path) as im:
#             return np.array(im.convert('RGB')).astype(np.float32)/255.0
#     def _to_tensor_norm(self, arr: np.ndarray) -> torch.Tensor:
#         t = torch.from_numpy(arr.transpose(2,0,1)).float()
#         return t*2-1
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         noisy_path = (self.root / row['noisy']).resolve()
#         clean_path = (self.root / row['clean']).resolve()
#         noisy = np.array(self.tf(Image.fromarray(_to_uint8(self._read_rgb(noisy_path)))))/255.0
#         clean = np.array(self.tf(Image.fromarray(_to_uint8(self._read_rgb(clean_path)))))/255.0
#         return {'noisy': self._to_tensor_norm(noisy), 'clean': self._to_tensor_norm(clean), 'path': str(clean_path)}
#
# mapping_csv = repo_root / 'Dataset' / 'dataset_mapping.csv'
# paired_ds = PairedMappingDataset(mapping_csv, repo_root)
# val_len = max(1, int(len(paired_ds) * cfg.val_split))
# train_len = len(paired_ds) - val_len
# train_ds, val_ds = random_split(paired_ds, [train_len, val_len], generator=torch.Generator().manual_seed(seed))
# train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
# val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
# print('Switched to paired dataset from mapping.')

In [ ]:
# Use more GPU memory: quick helpers to scale up batch size and rebuild loaders
def rebuild_loaders(new_batch_size: int | None = None, new_image_size: int | None = None):
    global full_ds, train_ds, val_ds, train_loader, val_loader, cfg
    if new_image_size is not None and new_image_size != cfg.image_size:
        cfg.image_size = new_image_size
        # Recreate dataset with new image size
        full_ds = CleanWithSyntheticNoiseDataset(clean_dir, image_size=cfg.image_size, use_albu=True)
        val_len = max(1, int(len(full_ds) * cfg.val_split))
        train_len = len(full_ds) - val_len
        train_ds, val_ds = random_split(full_ds, [train_len, val_len], generator=torch.Generator().manual_seed(seed))
    if new_batch_size is not None:
        cfg.batch_size = new_batch_size
    train_loader = _make_loader(train_ds, cfg.batch_size, True)
    val_loader   = _make_loader(val_ds,   cfg.batch_size, False)
    print(f"Rebuilt loaders -> batch_size={cfg.batch_size}, image_size={cfg.image_size}, workers={cfg.num_workers}")

# Example usage:
# rebuild_loaders(new_batch_size=16)            # double batch size
# rebuild_loaders(new_batch_size=16, new_image_size=512)  # bigger images too